In [ ]:
# In[1]:

import pandas as pd

# Load CSVs and parse timestamps as required (Unix seconds -> UTC)
metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

trace_df = pd.read_csv("trace.csv")
trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

log_df = pd.read_csv("log.csv")
log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

error_df = pd.read_csv("error_logs.csv")
error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# 1) metric.csv summary
metric_unique_cmdb = sorted(metric_df['cmdb_id'].dropna().unique().tolist())
metric_row_count = len(metric_df)
metric_top_combos = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 2) trace.csv summary
trace_unique_cmdb = sorted(trace_df['cmdb_id'].dropna().unique().tolist())
trace_row_count = len(trace_df)
trace_top_combos = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 3) log.csv summary
log_unique_cmdb = sorted(log_df['cmdb_id'].dropna().unique().tolist())
log_row_count = len(log_df)
log_top_combos = (
    log_df
    .groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
)

# 4) error_logs.csv summary
error_total_rows = len(error_df)
error_cmdb_counts = (
    error_df['cmdb_id']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'cmdb_id', 'cmdb_id': 'count'})
)
error_earliest_ts = error_df['timestamp'].min() if error_total_rows > 0 else None
error_latest_ts = error_df['timestamp'].max() if error_total_rows > 0 else None
error_top_messages = (
    error_df['message']
    .value_counts()
    .reset_index()
    .rename(columns={'index': 'message', 'message': 'count'})
    .head(10)
)

# Assemble compact summary dict (DataFrames kept for compact tabular presentation)
summary = {
    'metric': {
        'unique_cmdb_ids': metric_unique_cmdb,
        'row_count': metric_row_count,
        'top_cmdb_kpi_combinations': metric_top_combos[['cmdb_id', 'kpi_name', 'count']]
    },
    'trace': {
        'unique_cmdb_ids': trace_unique_cmdb,
        'row_count': trace_row_count,
        'top_cmdb_trace_combinations': trace_top_combos[['cmdb_id', 'trace_name', 'count']]
    },
    'log': {
        'unique_cmdb_ids': log_unique_cmdb,
        'row_count': log_row_count,
        'top_cmdb_log_combinations': log_top_combos[['cmdb_id', 'log_name', 'count']]
    },
    'error_logs': {
        'total_rows': error_total_rows,
        'cmdb_counts_top20': error_cmdb_counts.head(20),
        'earliest_timestamp_utc': error_earliest_ts,
        'latest_timestamp_utc': error_latest_ts,
        'top_messages': error_top_messages
    }
}

# Display the compact summary object
summary

```
Out[1]:
```
```python
# Compose a plain-English summary string based on the previously computed `summary` variable
metric_rows = summary['metric']['row_count']
metric_unique_count = len(summary['metric']['unique_cmdb_ids'])
metric_top_df = summary['metric']['top_cmdb_kpi_combinations']

trace_rows = summary['trace']['row_count']
trace_unique_count = len(summary['trace']['unique_cmdb_ids'])
trace_top_df = summary['trace']['top_cmdb_trace_combinations']

log_rows = summary['log']['row_count']
log_unique_count = len(summary['log']['unique_cmdb_ids'])
log_top_df = summary['log']['top_cmdb_log_combinations']

error_total = summary['error_logs']['total_rows']
error_earliest = summary['error_logs']['earliest_timestamp_utc']
error_latest = summary['error_logs']['latest_timestamp_utc']

def top_examples(df, key_col, val_col, n_items=3, max_vals=6):
    examples = []
    for cm in df[key_col].unique()[:n_items]:
        vals = df[df[key_col]==cm][val_col].unique()[:max_vals].tolist()
        examples.append(f"{cm}: {', '.join(vals)}")
    return "; ".join(examples)

metric_examples = top_examples(metric_top_df, 'cmdb_id', 'kpi_name', n_items=3)
trace_examples = top_examples(trace_top_df, 'cmdb_id', 'trace_name', n_items=3)
log_examples = top_examples(log_top_df, 'cmdb_id', 'log_name', n_items=6)

summary_text = (
    f"Telemetry summary (UTC):\n\n"
    f"1) metric.csv: {metric_rows} rows across {metric_unique_count} unique services/components. "
    f"Top (cmdb_id, kpi) combinations are dominated by many 25-row series. Examples: {metric_examples}.\n\n"
    f"2) trace.csv: {trace_rows} rows across {trace_unique_count} unique services/components. "
    f"Top trace combinations focus on travel-related services. Examples: {trace_examples}.\n\n"
    f"3) log.csv: {log_rows} rows across {log_unique_count} unique services/components. "
    f"Top log combos are 'log.error_count' and 'log.row_count' for many services. Examples: {log_examples}.\n\n"
    f"4) error_logs.csv: {error_total} rows (no error log records). "
    f"Earliest timestamp: {error_earliest}; Latest timestamp: {error_latest}.\n\n"
    f"Overall: metrics and traces show broad coverage across the fleet (notably travel and voucher services), "
    f"logs contain per-minute error_count/row_count series for many services, and there are no raw error_log entries to inspect."
)

summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'metric': {'unique_cmdb_ids': ['istio-init', 'ts-admin-basic-info-service', 'ts-admin-order-service', 'ts-admin-route-service', 'ts-admin-travel-service', 'ts-admin-user-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-avatar-service', 'ts-basic-service', 'ts-cancel-service', 'ts-config-mongo', 'ts-config-service', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-execute-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-mysql', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-news-service', 'ts-notification-mongo', 'ts-notification-service', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-payment-service', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-rebook-service', 'ts-route-mongo', 'ts-route-plan-service', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-station-service', 'ts-ticket-office-mongo', 'ts-ticket-office-service', 'ts-ticketinfo-service', 'ts-train-mongo', 'ts-train-service', 'ts-travel-mongo', 'ts-travel-plan-service', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo', 'ts-user-service', 'ts-verification-code-service', 'ts-voucher-mysql', 'ts-voucher-service'], 'row_count': 9009, 'top_cmdb_kpi_combinations':                          cmdb_id    kpi_name  count
362           ts-voucher-service      socket     25
361           ts-voucher-service         mem     25
360           ts-voucher-service      diskio     25
2    ts-admin-basic-info-service         cpu     25
3    ts-admin-basic-info-service      diskio     25
4    ts-admin-basic-info-service       error     25
5    ts-admin-basic-info-service  latency-50     25
6    ts-admin-basic-info-service  latency-90     25
7    ts-admin-basic-info-service         mem     25
321             ts-travel2-mongo         cpu     25
322             ts-travel2-mongo      diskio     25
323             ts-travel2-mongo         mem     25
324             ts-travel2-mongo      socket     25
325           ts-travel2-service         cpu     25
326           ts-travel2-service      diskio     25
327           ts-travel2-service  latency-50     25
328           ts-travel2-service  latency-90     25
313       ts-travel-plan-service      socket     25
314            ts-travel-service         cpu     25
315            ts-travel-service      diskio     25}, 'trace': {'unique_cmdb_ids': ['root', 'ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-service', 'ts-auth-service', 'ts-basic-service', 'ts-config-service', 'ts-consign-price-service', 'ts-consign-service', 'ts-contacts-service', 'ts-food-map-service', 'ts-food-service', 'ts-inside-payment-service', 'ts-order-other-service', 'ts-order-service', 'ts-payment-service', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-service', 'ts-route-service', 'ts-seat-service', 'ts-security-service', 'ts-station-service', 'ts-ticketinfo-service', 'ts-train-service', 'ts-travel-service', 'ts-travel2-service', 'ts-user-service'], 'row_count': 18584, 'top_cmdb_trace_combinations':                 cmdb_id                                        trace_name  count
728  ts-travel2-service  trace.from_ts-admin-travel-service.duration_mean     25
729  ts-travel2-service   trace.from_ts-admin-travel-service.duration_p95     25
730  ts-travel2-service     trace.from_ts-admin-travel-service.error_rate     25
731  ts-travel2-service      trace.from_ts-admin-travel-service.row_count     25
720   ts-travel-service          trace.to_ts-travel-service.duration_mean     25
721   ts-travel-service           trace.to_ts-travel-service.duration_p95     25
722   ts-travel-service             trace.to_ts-travel-service.error_rate     25
723   ts-travel-service              trace.to_ts-travel-service.row_count     25
724  ts-travel2-service                     trace.from_root.duration_mean     25
725  ts-travel2-service                      trace.from_root.duration_p95     25
726  ts-travel2-service                        trace.from_root.error_rate     25
727  ts-travel2-service                         trace.from_root.row_count     25
712   ts-travel-service      trace.to_ts-ticketinfo-service.duration_mean     25
713   ts-travel-service       trace.to_ts-ticketinfo-service.duration_p95     25
714   ts-travel-service         trace.to_ts-ticketinfo-service.error_rate     25
715   ts-travel-service          trace.to_ts-ticketinfo-service.row_count     25
716   ts-travel-service           trace.to_ts-train-service.duration_mean     25
717   ts-travel-service            trace.to_ts-train-service.duration_p95     25
718   ts-travel-service              trace.to_ts-train-service.error_rate     25
719   ts-travel-service               trace.to_ts-train-service.row_count     25}, 'log': {'unique_cmdb_ids': ['ts-admin-basic-info-service', 'ts-admin-travel-service', 'ts-assurance-mongo', 'ts-assurance-service', 'ts-auth-mongo', 'ts-auth-service', 'ts-basic-service', 'ts-config-mongo', 'ts-consign-mongo', 'ts-consign-price-mongo', 'ts-consign-service', 'ts-contacts-mongo', 'ts-contacts-service', 'ts-food-map-mongo', 'ts-food-map-service', 'ts-food-mongo', 'ts-food-service', 'ts-inside-payment-mongo', 'ts-inside-payment-service', 'ts-notification-mongo', 'ts-order-mongo', 'ts-order-other-mongo', 'ts-order-other-service', 'ts-order-service', 'ts-payment-mongo', 'ts-preserve-mongo', 'ts-preserve-other-mongo', 'ts-preserve-other-service', 'ts-preserve-service', 'ts-price-mongo', 'ts-price-service', 'ts-route-mongo', 'ts-route-service', 'ts-seat-service', 'ts-security-mongo', 'ts-security-service', 'ts-station-mongo', 'ts-ticket-office-mongo', 'ts-train-mongo', 'ts-travel-mongo', 'ts-travel-service', 'ts-travel2-mongo', 'ts-travel2-service', 'ts-ui-dashboard', 'ts-user-mongo'], 'row_count': 2188, 'top_cmdb_log_combinations':                         cmdb_id         log_name  count
0   ts-admin-basic-info-service  log.error_count     25
1   ts-admin-basic-info-service    log.row_count     25
2       ts-admin-travel-service  log.error_count     25
3       ts-admin-travel-service    log.row_count     25
21           ts-consign-service    log.row_count     25
20           ts-consign-service  log.error_count     25
13             ts-basic-service    log.row_count     25
12             ts-basic-service  log.error_count     25
11              ts-auth-service    log.row_count     25
10              ts-auth-service  log.error_count     25
61             ts-price-service    log.row_count     25
60             ts-price-service  log.error_count     25
64             ts-route-service  log.error_count     25
65             ts-route-service    log.row_count     25
67              ts-seat-service    log.row_count     25
66              ts-seat-service  log.error_count     25
47             ts-order-service    log.row_count     25
46             ts-order-service  log.error_count     25
45       ts-order-other-service    log.row_count     25
44       ts-order-other-service  log.error_count     25}, 'error_logs': {'total_rows': 0, 'cmdb_counts_top20': Empty DataFrame
Columns: [count, count]
Index: [], 'earliest_timestamp_utc': None, 'latest_timestamp_utc': None, 'top_messages': Empty DataFrame
Columns: [count, count]
Index: []}}```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# reuse metric_df if available, otherwise load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp is parsed correctly (Unix seconds -> UTC)
if metric_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Compute global aggregates (over entire series) per (cmdb_id, kpi_name)
group_keys = ['cmdb_id', 'kpi_name']
global_counts = metric_df.groupby(group_keys)['value'].count().rename('total_points')
global_p95 = metric_df.groupby(group_keys)['value'].quantile(0.95).rename('p95_value')

global_df = pd.concat([global_counts, global_p95], axis=1).reset_index()

# 2) Define incident window (UTC)
window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)

# 3) Filter metric data to the incident window
metric_window = metric_df[(metric_df['timestamp'] >= window_start) & (metric_df['timestamp'] <= window_end)].copy()

# 4) Merge global p95 into the windowed rows for exceed checks
metric_window = metric_window.merge(global_df[['cmdb_id', 'kpi_name', 'p95_value']], on=group_keys, how='left')
metric_window['exceed'] = metric_window['value'] > metric_window['p95_value']

# 5) Compute window-level aggregations
window_points = metric_window.groupby(group_keys)['value'].count().rename('window_points')
max_value_in_window = metric_window.groupby(group_keys)['value'].max().rename('max_value_in_window')

# 6) Compute exceed-specific aggregations
df_exceed = metric_window[metric_window['exceed']]
count_exceed = df_exceed.groupby(group_keys).size().rename('count_exceed')
earliest_breach = df_exceed.groupby(group_keys)['timestamp'].min().rename('earliest_breach_utc')

# 7) Assemble final table by joining global and window aggregations
result = (
    global_df
    .merge(window_points.reset_index(), on=group_keys, how='left')
    .merge(max_value_in_window.reset_index(), on=group_keys, how='left')
    .merge(count_exceed.reset_index(), on=group_keys, how='left')
    .merge(earliest_breach.reset_index(), on=group_keys, how='left')
)

# Fill NaNs for groups with no window rows / no exceeds
result['window_points'] = result['window_points'].fillna(0).astype(int)
result['count_exceed'] = result['count_exceed'].fillna(0).astype(int)
result['max_value_in_window'] = result['max_value_in_window'].where(result['window_points']>0, other=np.nan)
result['earliest_breach_utc'] = result['earliest_breach_utc'].where(result['count_exceed']>0, other=pd.NaT)

# Filter to groups with count_exceed > 0, sort and limit to top 20
df_out = (
    result[result['count_exceed'] > 0]
    .sort_values('count_exceed', ascending=False)
    .reset_index(drop=True)
)

# Select and order the requested columns
df_out = df_out[['cmdb_id', 'kpi_name', 'total_points', 'p95_value', 'window_points', 'count_exceed', 'earliest_breach_utc', 'max_value_in_window']]

# Limit to top 20 as requested
df_out = df_out.head(20)

# Display the compact result DataFrame (JSON-like table/dict)
df_out

```
Out[2]:
```
```python
# Build a concise plain-English summary from the existing `df_out` result
try:
    df_out
except NameError:
    # Safety: reload and recompute if df_out is not in memory (shouldn't be necessary normally)
    import pandas as pd, numpy as np
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
    group_keys = ['cmdb_id','kpi_name']
    global_df = pd.concat([
        metric_df.groupby(group_keys)['value'].count().rename('total_points'),
        metric_df.groupby(group_keys)['value'].quantile(0.95).rename('p95_value')
    ], axis=1).reset_index()
    window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
    window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)
    metric_window = metric_df[(metric_df['timestamp'] >= window_start) & (metric_df['timestamp'] <= window_end)].copy()
    metric_window = metric_window.merge(global_df[['cmdb_id','kpi_name','p95_value']], on=group_keys, how='left')
    metric_window['exceed'] = metric_window['value'] > metric_window['p95_value']
    window_points = metric_window.groupby(group_keys)['value'].count().rename('window_points')
    max_value_in_window = metric_window.groupby(group_keys)['value'].max().rename('max_value_in_window')
    df_exceed = metric_window[metric_window['exceed']]
    count_exceed = df_exceed.groupby(group_keys).size().rename('count_exceed')
    earliest_breach = df_exceed.groupby(group_keys)['timestamp'].min().rename('earliest_breach_utc')
    result = (
        global_df
        .merge(window_points.reset_index(), on=group_keys, how='left')
        .merge(max_value_in_window.reset_index(), on=group_keys, how='left')
        .merge(count_exceed.reset_index(), on=group_keys, how='left')
        .merge(earliest_breach.reset_index(), on=group_keys, how='left')
    )
    result['window_points'] = result['window_points'].fillna(0).astype(int)
    result['count_exceed'] = result['count_exceed'].fillna(0).astype(int)
    result['max_value_in_window'] = result['max_value_in_window'].where(result['window_points']>0, other=pd.NA)
    result['earliest_breach_utc'] = result['earliest_breach_utc'].where(result['count_exceed']>0, other=pd.NaT)
    df_out = result[result['count_exceed'] > 0].sort_values('count_exceed', ascending=False).reset_index(drop=True)
    df_out = df_out[['cmdb_id','kpi_name','total_points','p95_value','window_points','count_exceed','earliest_breach_utc','max_value_in_window']].head(20)

# Compose summary
num_groups = len(df_out)
if num_groups == 0:
    summary = "No (cmdb_id, kpi_name) groups exceeded their global P95 in the incident window."
else:
    # Note general pattern observed in results
    all_total_points = df_out['total_points'].unique()
    all_window_points = df_out['window_points'].unique()
    all_count_exceed = df_out['count_exceed'].unique()
    pattern_notes = []
    if len(all_total_points)==1:
        pattern_notes.append(f"total_points={int(all_total_points[0])}")
    if len(all_window_points)==1:
        pattern_notes.append(f"window_points={int(all_window_points[0])}")
    if len(all_count_exceed)==1:
        pattern_notes.append(f"count_exceed={int(all_count_exceed[0])}")
    pattern_summary = ", ".join(pattern_notes) if pattern_notes else ""
    
    # Prepare example lines (up to first 10)
    lines = []
    for i, row in df_out.head(10).iterrows():
        cm = row['cmdb_id']
        kpi = row['kpi_name']
        ce = int(row['count_exceed'])
        eb = row['earliest_breach_utc']
        eb_str = eb.isoformat() if pd.notna(eb) else "None"
        mv = row['max_value_in_window']
        # format float compactly
        mv_str = f"{float(mv):.6g}" if pd.notna(mv) else "None"
        lines.append(f"{i+1}) {cm} | {kpi} — breaches: {ce}, earliest: {eb_str}, max_in_window: {mv_str}")
    
    summary = (
        f"Found {num_groups} (cmdb_id, kpi_name) groups that exceeded their global P95 during the incident window.\n"
        f"{'Common pattern: ' + pattern_summary + '. ' if pattern_summary else ''}"
        f"Top examples:\n" + "\n".join(lines)
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points     p95_value  window_points  count_exceed       earliest_breach_utc  max_value_in_window
0            ts-voucher-service         cpu            25  3.617299e-02             25             2 2024-01-24 07:53:00+00:00         3.659472e-02
1   ts-admin-basic-info-service         cpu            25  2.128417e+00             25             2 2024-01-24 07:31:00+00:00         3.107765e+00
2   ts-admin-basic-info-service      diskio            25  6.550358e+04             25             2 2024-01-24 07:30:00+00:00         6.641074e+04
3   ts-admin-basic-info-service  latency-50            25  2.295604e-02             25             2 2024-01-24 07:30:00+00:00         5.285965e-02
4   ts-admin-basic-info-service  latency-90            25  5.570275e-02             25             2 2024-01-24 07:30:00+00:00         1.491683e-01
5   ts-admin-basic-info-service         mem            25  2.272049e+08             25             2 2024-01-24 07:53:00+00:00         2.275082e+08
6              ts-voucher-mysql         cpu            25  5.061772e-02             25             2 2024-01-24 07:44:00+00:00         5.164948e-02
7   ts-admin-basic-info-service    workload            25  6.774100e-01             25             2 2024-01-24 07:31:00+00:00         7.132000e-01
8        ts-admin-order-service         cpu            25  1.291733e-01             25             2 2024-01-24 07:32:00+00:00         1.647479e-01
9        ts-admin-order-service      diskio            25  5.253726e+04             25             2 2024-01-24 07:33:00+00:00         6.940041e+04
10       ts-admin-order-service         mem            25  1.919022e+08             25             2 2024-01-24 07:50:00+00:00         1.919029e+08
11       ts-admin-order-service      socket            25  3.366667e+00             25             2 2024-01-24 07:46:00+00:00         3.450000e+00
12       ts-admin-route-service         cpu            25  1.269933e-01             25             2 2024-01-24 07:32:00+00:00         1.390137e-01
13       ts-admin-route-service      diskio            25  6.544297e+04             25             2 2024-01-24 07:32:00+00:00         8.056620e+04
14       ts-admin-route-service         mem            25  1.971672e+08             25             2 2024-01-24 07:53:00+00:00         1.971707e+08
15       ts-admin-route-service      socket            25  3.263333e+00             25             2 2024-01-24 07:46:00+00:00         3.300000e+00
16      ts-admin-travel-service         cpu            25  1.343998e+00             25             2 2024-01-24 07:31:00+00:00         1.430398e+00
17      ts-admin-travel-service      diskio            25  6.470577e+04             25             2 2024-01-24 07:30:00+00:00         7.749782e+04
18      ts-admin-travel-service  latency-50            25  1.044992e+00             25             2 2024-01-24 07:30:00+00:00         1.115859e+00
19      ts-admin-travel-service  latency-90            25  2.136199e+00             25             2 2024-01-24 07:43:00+00:00         2.186384e+00```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load and parse timestamps
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Ensure timestamp dtype is UTC datetime
if metric_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Candidate components
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# Filter metric rows to only candidate components for global calculations
metric_cand = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Compute global total_points and P95 per (cmdb_id, kpi_name) using the entire series
group_keys = ['cmdb_id', 'kpi_name']
global_stats = (
    metric_cand
    .groupby(group_keys)['value']
    .agg(total_points='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# Define incident window (UTC)
window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)

# Filter metric data to the incident window (only candidates)
metric_window = metric_cand[(metric_cand['timestamp'] >= window_start) & (metric_cand['timestamp'] <= window_end)].copy()

# Merge global p95 info into windowed rows
metric_window = metric_window.merge(global_stats, on=group_keys, how='left')

# Determine exceedances (value > p95_value)
metric_window['exceed'] = metric_window['value'] > metric_window['p95_value']

# Compute window_points and max_value_in_window per group (within window)
window_agg = (
    metric_window
    .groupby(group_keys)
    .agg(window_points=('value', 'count'),
         max_value_in_window=('value', 'max'))
    .reset_index()
)

# Extract breach rows and aggregate breach info
breach_rows = metric_window[metric_window['exceed']].copy()
if not breach_rows.empty:
    breach_agg = (
        breach_rows
        .sort_values(['cmdb_id','kpi_name','timestamp'])
        .groupby(group_keys)
        .agg(
            count_exceed=('exceed', 'size'),
            breach_timestamps_utc=('timestamp', lambda s: s.dt.strftime('%Y-%m-%dT%H:%M:%SZ').tolist()[:10]),
            earliest_breach_utc=('timestamp', 'min'),
        )
        .reset_index()
    )
else:
    breach_agg = pd.DataFrame(columns=group_keys + ['count_exceed', 'breach_timestamps_utc', 'earliest_breach_utc'])

# Assemble final summary by joining global_stats, window_agg, and breach_agg
summary = (
    global_stats
    .merge(window_agg, on=group_keys, how='left')
    .merge(breach_agg, on=group_keys, how='left')
)

# Fill NaNs for window counts
summary['window_points'] = summary['window_points'].fillna(0).astype(int)
summary['max_value_in_window'] = summary['max_value_in_window']  # keep NaN if no window points

# Keep only groups with count_exceed > 0
summary['count_exceed'] = summary['count_exceed'].fillna(0).astype(int)
summary = summary[summary['count_exceed'] > 0].copy()

# Format earliest_breach_utc as ISO strings (or None)
summary['earliest_breach_utc'] = summary['earliest_breach_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Compute percent_over_p95: (max_value_in_window - p95_value)/p95_value*100
def compute_pct(row):
    p95 = row['p95_value']
    mv = row['max_value_in_window']
    if pd.isna(p95) or pd.isna(mv):
        return np.nan
    if p95 == 0:
        if mv > 0:
            return np.inf
        else:
            return 0.0
    return (mv - p95) / p95 * 100.0

summary['percent_over_p95'] = summary.apply(compute_pct, axis=1)

# Ensure breach_timestamps_utc column exists and is list of ISO strings (limit to first 10 already handled)
if 'breach_timestamps_utc' not in summary.columns:
    summary['breach_timestamps_utc'] = [[] for _ in range(len(summary))]

# Select and order columns as requested
out_cols = ['cmdb_id', 'kpi_name', 'total_points', 'p95_value', 'window_points',
            'count_exceed', 'breach_timestamps_utc', 'earliest_breach_utc',
            'max_value_in_window', 'percent_over_p95']

df_out = summary[out_cols].copy()

# Sort by percent_over_p95 desc then count_exceed desc, limit to top 20
df_out = df_out.sort_values(by=['percent_over_p95', 'count_exceed'], ascending=[False, False]).reset_index(drop=True).head(20)

# Keep the output compact (DataFrame). Display df_out.
df_out

```
Out[3]:
```
```python
# Compose a plain-English summary from the existing `df_out` (candidate-focused) result
try:
    df_out
except NameError:
    raise RuntimeError("df_out not found in session. Please run the previous analysis cell first.")

top_rows = df_out.head(5)
summary_lines = [
    "Summary of metric breaches for candidate components during 2024-01-24 07:27:12 to 07:57:12 UTC:",
    "",
    f"- Total groups with P95 breaches (among the 5 candidates): {len(df_out)}.",
    "- All reported groups had 2 exceedances within the incident window.",
    "",
    "Top observations (by percent over global P95):"
]

# Add top 5 detailed highlights
for i, row in top_rows.iterrows():
    summary_lines.append(
        f"  {i+1}. {row['cmdb_id']} | {row['kpi_name']}: "
        f"max_in_window={float(row['max_value_in_window']):.6g}, "
        f"p95={float(row['p95_value']):.6g}, "
        f"percent_over_p95={float(row['percent_over_p95']):.1f}%, "
        f"breaches_at={row['breach_timestamps_utc']}"
    )

summary_lines += [
    "",
    "Notable patterns:",
    "- ts-train-service shows the largest relative spikes in latency (latency-90 ~261% over P95, latency-50 ~150% over P95) and elevated CPU/disk I/O.",
    "- ts-travel-service also shows large latency and CPU spikes (latency-50 ~119% over P95; CPU ~54% over P95).",
    "- ts-order-service, ts-route-service, and ts-auth-service each have multiple KPIs (latency, CPU, socket, diskio, memory/workload) exceeding their global P95s, though with smaller percent increases than the top train/travel entries.",
    "- Many breaches occur around 07:30–07:33 UTC (several groups list 2024-01-24T07:30:00Z and 07:31:00Z as earliest breaches), indicating a correlated spike across services during that interval.",
    "",
    "Conclusion:",
    "Multiple components (especially ts-train-service and ts-travel-service) experienced significant latency and resource spikes during the incident window. The telemetry points to elevated latency and increased CPU/disk I/O in those services as primary anomalies to investigate further."
]

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  total_points     p95_value  window_points  count_exceed                         breach_timestamps_utc   earliest_breach_utc  max_value_in_window  percent_over_p95
0    ts-train-service  latency-90            25  2.535307e-02             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         9.161798e-02        261.368366
1    ts-train-service  latency-50            25  8.827814e-03             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         2.210917e-02        150.448939
2   ts-travel-service  latency-50            25  5.405642e-02             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         1.181642e-01        118.594186
3    ts-train-service         cpu            25  1.159766e+01             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         1.928361e+01         66.271586
4   ts-travel-service         cpu            25  1.628450e+01             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         2.500746e+01         53.566031
5    ts-train-service      diskio            25  6.419425e+04             25             2  [2024-01-24T07:32:00Z, 2024-01-24T07:44:00Z]  2024-01-24T07:32:00Z         9.048346e+04         40.952593
6    ts-order-service  latency-50            25  1.270841e-02             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         1.785731e-02         40.515688
7    ts-route-service  latency-90            25  1.073206e-01             25             2  [2024-01-24T07:49:00Z, 2024-01-24T07:52:00Z]  2024-01-24T07:49:00Z         1.345506e-01         25.372547
8   ts-travel-service      diskio            25  7.854216e+04             25             2  [2024-01-24T07:32:00Z, 2024-01-24T07:33:00Z]  2024-01-24T07:32:00Z         9.544408e+04         21.519544
9    ts-order-service         cpu            25  8.481280e+00             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:32:00Z]  2024-01-24T07:30:00Z         9.892041e+00         16.633831
10    ts-auth-service  latency-90            25  9.213969e-01             25             2  [2024-01-24T07:35:00Z, 2024-01-24T07:48:00Z]  2024-01-24T07:35:00Z         1.059670e+00         15.006886
11   ts-route-service  latency-50            25  1.655512e-02             25             2  [2024-01-24T07:45:00Z, 2024-01-24T07:46:00Z]  2024-01-24T07:45:00Z         1.851285e-02         11.825532
12   ts-order-service  latency-90            25  5.490229e-02             25             2  [2024-01-24T07:30:00Z, 2024-01-24T07:31:00Z]  2024-01-24T07:30:00Z         6.048100e-02         10.161177
13   ts-order-service      socket            25  1.245667e+01             25             2  [2024-01-24T07:43:00Z, 2024-01-24T07:47:00Z]  2024-01-24T07:43:00Z         1.340000e+01          7.572919
14    ts-auth-service      socket            25  1.529744e+01             25             2  [2024-01-24T07:48:00Z, 2024-01-24T07:49:00Z]  2024-01-24T07:48:00Z         1.625000e+01          6.226953
15   ts-order-service      diskio            25  1.865106e+06             25             2  [2024-01-24T07:45:00Z, 2024-01-24T07:50:00Z]  2024-01-24T07:45:00Z         1.935933e+06          3.797512
16   ts-route-service      diskio            25  4.303303e+07             25             2  [2024-01-24T07:45:00Z, 2024-01-24T07:46:00Z]  2024-01-24T07:45:00Z         4.459684e+07          3.633996
17   ts-order-service    workload            25  6.994423e+00             25             2  [2024-01-24T07:43:00Z, 2024-01-24T07:46:00Z]  2024-01-24T07:43:00Z         7.232200e+00          3.399518
18    ts-auth-service         cpu            25  1.670402e+01             25             2  [2024-01-24T07:33:00Z, 2024-01-24T07:51:00Z]  2024-01-24T07:33:00Z         1.722101e+01          3.094976
19   ts-route-service         mem            25  7.124868e+08             25             2  [2024-01-24T07:46:00Z, 2024-01-24T07:50:00Z]  2024-01-24T07:46:00Z         7.340014e+08          3.019656```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse trace_df if available; otherwise load and parse timestamps
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Ensure timestamp is datetime UTC
if trace_df['timestamp'].dtype != 'datetime64[ns, UTC]':
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# 1) Global P95 per (cmdb_id, trace_name) over the full trace series
group_keys = ['cmdb_id', 'trace_name']
global_stats = (
    trace_df
    .groupby(group_keys)['value']
    .agg(total_points='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# 2) Incident window filter (UTC)
window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)
trace_window = trace_df[(trace_df['timestamp'] >= window_start) & (trace_df['timestamp'] <= window_end)].copy()

# 3) Merge global P95 into window rows and compute exceedances
trace_window = trace_window.merge(global_stats, on=group_keys, how='left')
trace_window['exceed'] = trace_window['value'] > trace_window['p95_value']

# Window-level aggregates
window_agg = (
    trace_window
    .groupby(group_keys)
    .agg(window_points=('value', 'count'),
         max_value_in_window=('value', 'max'))
    .reset_index()
)

# Breach-specific aggregates
breach_rows = trace_window[trace_window['exceed']].copy()
if not breach_rows.empty:
    breach_agg = (
        breach_rows
        .groupby(group_keys)
        .agg(count_exceed=('exceed', 'size'),
             earliest_breach_utc=('timestamp', 'min'))
        .reset_index()
    )
else:
    breach_agg = pd.DataFrame(columns=group_keys + ['count_exceed', 'earliest_breach_utc'])

# Assemble final summary
trace_summary = (
    global_stats
    .merge(window_agg, on=group_keys, how='left')
    .merge(breach_agg, on=group_keys, how='left')
)

# Fill defaults
trace_summary['window_points'] = trace_summary['window_points'].fillna(0).astype(int)
trace_summary['count_exceed'] = trace_summary['count_exceed'].fillna(0).astype(int)
trace_summary['max_value_in_window'] = trace_summary['max_value_in_window']  # keep NaN if no window points
trace_summary['earliest_breach_utc'] = trace_summary['earliest_breach_utc']  # keep NaT if none

# Keep only groups with count_exceed > 0
trace_summary = trace_summary[trace_summary['count_exceed'] > 0].copy()

# Format earliest_breach_utc as ISO strings
trace_summary['earliest_breach_utc'] = trace_summary['earliest_breach_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Compute percent_over_p95 with handling for p95_value == 0
def pct_over_p95(row):
    p95 = row['p95_value']
    mv = row['max_value_in_window']
    if pd.isna(p95) or pd.isna(mv):
        return np.nan
    if p95 == 0.0:
        return np.inf if mv > 0 else 0.0
    return (mv - p95) / p95 * 100.0

trace_summary['percent_over_p95'] = trace_summary.apply(pct_over_p95, axis=1)

# Select and order requested columns
out_cols = ['cmdb_id', 'trace_name', 'total_points', 'p95_value', 'window_points',
            'count_exceed', 'earliest_breach_utc', 'max_value_in_window', 'percent_over_p95']
trace_summary = trace_summary[out_cols]

# 4) Sort by percent_over_p95 desc then count_exceed desc, limit to top 20
trace_summary = trace_summary.sort_values(['percent_over_p95', 'count_exceed'], ascending=[False, False]).reset_index(drop=True).head(20)

# Additional: compact breakdown of trace_name patterns for first 10 returned rows
def classify_trace_name(tn):
    tn_lower = str(tn).lower()
    if 'from_' in tn_lower or '.from' in tn_lower or 'trace.from' in tn_lower:
        return 'from_X'
    if 'to_' in tn_lower or '.to' in tn_lower or 'trace.to' in tn_lower:
        return 'to_X'
    return 'other'

trace_patterns = trace_summary[['trace_name']].head(10).copy()
trace_patterns['pattern'] = trace_patterns['trace_name'].apply(classify_trace_name)

# Keep outputs compact: trace_summary (top results) and trace_patterns (first 10 patterns)
trace_summary, trace_patterns

```
Out[4]:
```
```python
# Plain-English summary string of the trace analysis results
summary = (
    "Trace analysis (2024-01-24 07:27:12 to 07:57:12 UTC):\n\n"
    "1) Multiple trace groups exceeded their global P95 during the incident window (top 20 shown).\n"
    "2) The largest relative spikes are in p95 durations for inter-service calls involving station and auth: "
    "e.g., trace.to_ts-station-service.duration_p95 (ts-food-service → ts-station-service) rose ~979% over global P95, "
    "and trace.from/to_ts-auth-service.duration_p95 rose ~949% over P95.\n"
    "3) Other notable spikes (hundreds of percent over P95) include calls related to preserve, consign, contacts, and order-other flows.\n"
    "4) Breaches cluster in two main intervals: around 07:30:00Z (many groups) and again around 07:45–07:49Z for another set of groups.\n"
    "5) Pattern breakdown: most returned trace_names are 'to_X' or 'from_X' styles (inter-service call latencies), indicating cross-service latency spikes rather than a single internal metric.\n"
    "6) Among the candidate services, ts-auth-service shows strong trace anomalies (both from_* and to_* duration_p95 spikes). ts-route-service appears via traces from ts-basic-service, while ts-order-related traces (order-other) also show large increases.\n"
    "7) Interpretation / recommendation: traces point to widespread request latency between services (notably calls to/from station and auth) during the incident window. Investigate networking, upstream dependencies, and the station/auth service stacks around 07:30Z and 07:45–07:49Z as likely contributors."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                      cmdb_id                                         trace_name  total_points  p95_value  window_points  count_exceed   earliest_breach_utc  max_value_in_window  percent_over_p95
0             ts-food-service           trace.to_ts-station-service.duration_p95            25   0.007988             25             2  2024-01-24T07:45:00Z             0.086224        979.367355
1          ts-station-service            trace.from_ts-food-service.duration_p95            25   0.007988             25             2  2024-01-24T07:45:00Z             0.086224        979.367355
2             ts-auth-service            trace.from_ts-auth-service.duration_p95            25   0.005860             25             2  2024-01-24T07:47:00Z             0.061479        949.210516
3             ts-auth-service              trace.to_ts-auth-service.duration_p95            25   0.005860             25             2  2024-01-24T07:47:00Z             0.061479        949.210516
4                        root          trace.to_ts-preserve-service.duration_p95            24   1.826232             24             2  2024-01-24T07:30:00Z             8.171924        347.474454
5         ts-preserve-service                       trace.from_root.duration_p95            24   1.826232             24             2  2024-01-24T07:30:00Z             8.171924        347.474454
6    ts-consign-price-service         trace.from_ts-consign-service.duration_p95            25   0.019819             25             2  2024-01-24T07:30:00Z             0.080347        305.412469
7          ts-consign-service     trace.to_ts-consign-price-service.duration_p95            25   0.019819             25             2  2024-01-24T07:30:00Z             0.080347        305.412469
8      ts-order-other-service           trace.to_ts-station-service.duration_p95            25   0.103136             25             2  2024-01-24T07:49:00Z             0.382606        270.972875
9          ts-station-service     trace.from_ts-order-other-service.duration_p95            25   0.103136             25             2  2024-01-24T07:49:00Z             0.382606        270.972875
10        ts-contacts-service        trace.from_ts-contacts-service.duration_p95            24   0.019077             24             2  2024-01-24T07:30:00Z             0.062925        229.847676
11        ts-contacts-service          trace.to_ts-contacts-service.duration_p95            24   0.019077             24             2  2024-01-24T07:30:00Z             0.062925        229.847676
12  ts-preserve-other-service         trace.to_ts-security-service.duration_mean            22   0.061642             22             2  2024-01-24T07:31:00Z             0.199913        224.312442
13        ts-security-service  trace.from_ts-preserve-other-service.duration_...            22   0.061642             22             2  2024-01-24T07:31:00Z             0.199913        224.312442
14                       root         trace.to_ts-preserve-service.duration_mean            24   1.581043             24             2  2024-01-24T07:30:00Z             4.788820        202.889905
15        ts-preserve-service                      trace.from_root.duration_mean            24   1.581043             24             2  2024-01-24T07:30:00Z             4.788820        202.889905
16           ts-basic-service             trace.to_ts-route-service.duration_p95            25   0.087537             25             2  2024-01-24T07:50:00Z             0.245240        180.156586
17           ts-route-service           trace.from_ts-basic-service.duration_p95            25   0.087537             25             2  2024-01-24T07:50:00Z             0.245240        180.156586
18                       root       trace.to_ts-order-other-service.duration_p95            25   0.151934             25             2  2024-01-24T07:36:00Z             0.393462        158.969072
19     ts-order-other-service                       trace.from_root.duration_p95            25   0.151934             25             2  2024-01-24T07:36:00Z             0.393462        158.969072,                                        trace_name pattern
0        trace.to_ts-station-service.duration_p95    to_X
1         trace.from_ts-food-service.duration_p95  from_X
2         trace.from_ts-auth-service.duration_p95  from_X
3           trace.to_ts-auth-service.duration_p95    to_X
4       trace.to_ts-preserve-service.duration_p95    to_X
5                    trace.from_root.duration_p95  from_X
6      trace.from_ts-consign-service.duration_p95  from_X
7  trace.to_ts-consign-price-service.duration_p95    to_X
8        trace.to_ts-station-service.duration_p95    to_X
9  trace.from_ts-order-other-service.duration_p95  from_X)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing loaded DataFrames if present; otherwise load and parse timestamps
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")
    log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsing
    if log_df['timestamp'].dtype != 'datetime64[ns, UTC]':
        log_df['timestamp'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

try:
    error_df
except NameError:
    error_df = pd.read_csv("error_logs.csv")
    error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)
else:
    if error_df['timestamp'].dtype != 'datetime64[ns, UTC]':
        error_df['timestamp'] = pd.to_datetime(error_df['timestamp'], unit='s', utc=True)

# Candidate services
candidates = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1 & 2) Compute global P95 per (cmdb_id, log_name) over full series for candidate services
log_cand = log_df[log_df['cmdb_id'].isin(candidates)].copy()
group_keys = ['cmdb_id', 'log_name']
global_stats = (
    log_cand
    .groupby(group_keys)['value']
    .agg(total_points='count', p95_value=lambda x: x.quantile(0.95))
    .reset_index()
)

# 3) Filter to incident window
window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)
log_window = log_cand[(log_cand['timestamp'] >= window_start) & (log_cand['timestamp'] <= window_end)].copy()

# 4) Merge global p95 into window rows and compute exceedances
log_window = log_window.merge(global_stats, on=group_keys, how='left')
log_window['exceed'] = log_window['value'] > log_window['p95_value']

# Window-level aggregations per (cmdb_id, log_name)
window_agg = (
    log_window
    .groupby(group_keys)
    .agg(window_points=('value', 'count'),
         sum_value_in_window=('value', 'sum'),
         max_value_in_window=('value', 'max'))
    .reset_index()
)

# Breach-specific aggregates
breach_rows = log_window[log_window['exceed']].copy()
if not breach_rows.empty:
    breach_agg = (
        breach_rows
        .groupby(group_keys)
        .agg(count_exceed=('exceed', 'size'),
             earliest_breach_utc=('timestamp', 'min'))
        .reset_index()
    )
else:
    breach_agg = pd.DataFrame(columns=group_keys + ['count_exceed', 'earliest_breach_utc'])

# Assemble group-level summary
group_summary = (
    global_stats
    .merge(window_agg, on=group_keys, how='left')
    .merge(breach_agg, on=group_keys, how='left')
)

# Fill defaults and format timestamps
group_summary['window_points'] = group_summary['window_points'].fillna(0).astype(int)
group_summary['sum_value_in_window'] = group_summary['sum_value_in_window'].fillna(0)
group_summary['max_value_in_window'] = group_summary['max_value_in_window']  # keep NaN if no window rows
group_summary['count_exceed'] = group_summary['count_exceed'].fillna(0).astype(int)
group_summary['earliest_breach_utc'] = group_summary['earliest_breach_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Keep only groups with count_exceed > 0
group_summary = group_summary[group_summary['count_exceed'] > 0].copy()

# Select and order requested columns
group_summary = group_summary[['cmdb_id','log_name','total_points','p95_value',
                               'window_points','count_exceed','earliest_breach_utc',
                               'sum_value_in_window','max_value_in_window']]

# 5) Per-service aggregated window-level sums for the five candidates
# sum_log_error_count, sum_log_row_count, earliest_nonzero_error_count_utc
# Compute sums by filtering log_window by log_name
error_count_rows = log_window[log_window['log_name'] == 'log.error_count'].copy()
row_count_rows = log_window[log_window['log_name'] == 'log.row_count'].copy()

svc_error_sum = error_count_rows.groupby('cmdb_id')['value'].sum().rename('sum_log_error_count')
svc_row_sum = row_count_rows.groupby('cmdb_id')['value'].sum().rename('sum_log_row_count')

# earliest nonzero error_count timestamp per service
if not error_count_rows.empty:
    nz = error_count_rows[error_count_rows['value'] > 0].copy()
    if not nz.empty:
        earliest_nz = nz.groupby('cmdb_id')['timestamp'].min().rename('earliest_nonzero_error_count_utc')
    else:
        earliest_nz = pd.Series(dtype='datetime64[ns, UTC]', name='earliest_nonzero_error_count_utc')
else:
    earliest_nz = pd.Series(dtype='datetime64[ns, UTC]', name='earliest_nonzero_error_count_utc')

# Assemble service-level DataFrame for all candidates (ensure all candidates present)
svc_df = pd.DataFrame({'cmdb_id': candidates})
svc_df = svc_df.set_index('cmdb_id')
svc_df = svc_df.join(svc_error_sum).join(svc_row_sum).join(earliest_nz)
svc_df = svc_df.fillna({'sum_log_error_count': 0, 'sum_log_row_count': 0})
# Format earliest timestamp to ISO or None
svc_df['earliest_nonzero_error_count_utc'] = svc_df['earliest_nonzero_error_count_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
svc_df['earliest_nonzero_error_count_utc'] = svc_df['earliest_nonzero_error_count_utc'].where(pd.notna(svc_df['earliest_nonzero_error_count_utc']), None)
svc_df = svc_df.reset_index()

# 6) Sort group_summary by count_exceed desc then sum_value_in_window desc, limit top 20
group_summary = group_summary.sort_values(['count_exceed','sum_value_in_window'], ascending=[False, False]).reset_index(drop=True).head(20)

# Error logs summary: total rows and top 5 messages
error_total_rows = len(error_df)
if error_total_rows > 0:
    top_messages = (
        error_df['message']
        .value_counts()
        .head(5)
        .reset_index()
        .rename(columns={'index': 'message', 'message': 'count'})
    )
else:
    top_messages = pd.DataFrame(columns=['message','count'])

# Assemble final compact result dict
result = {
    'group_summary': group_summary,            # DataFrame: per (cmdb_id, log_name) groups with breaches
    'service_aggregates': svc_df,              # DataFrame: per-candidate aggregated window sums
    'error_logs': {
        'total_rows': error_total_rows,
        'top_messages': top_messages          # DataFrame with top 5 messages and counts (may be empty)
    }
}

result

```
Out[5]:
```
```python
# Compose a plain-English summary from the previously computed `result` dict
gs = result['group_summary']
svc = result['service_aggregates']
err = result['error_logs']

lines = []
lines.append("Log analysis summary for candidate services during 2024-01-24 07:27:12 to 07:57:12 UTC:")
lines.append("")

# Group-level breaches
if gs.shape[0] == 0:
    lines.append("- No (cmdb_id, log_name) groups exceeded their global P95 in the window.")
else:
    lines.append(f"- {len(gs)} (cmdb_id, log_name) groups exceeded global P95. Top results:")
    for i, row in gs.iterrows():
        lines.append(
            f"  {i+1}. {row['cmdb_id']} | {row['log_name']}: "
            f"p95={row['p95_value']}, window_points={int(row['window_points'])}, "
            f"count_exceed={int(row['count_exceed'])}, earliest_breach={row['earliest_breach_utc']}, "
            f"sum_in_window={int(row['sum_value_in_window'])}, max_in_window={int(row['max_value_in_window'])}"
        )

lines.append("")

# Per-service aggregated window sums
lines.append("- Per-service window aggregates (log.error_count sum, log.row_count sum, earliest nonzero error_count):")
for _, s in svc.iterrows():
    lines.append(
        f"  {s['cmdb_id']}: sum_log_error_count={int(s['sum_log_error_count'])}, "
        f"sum_log_row_count={int(s['sum_log_row_count'])}, "
        f"earliest_nonzero_error_count_utc={s['earliest_nonzero_error_count_utc']}"
    )

lines.append("")

# Error logs summary
lines.append(f"- error_logs.csv: total_rows = {err['total_rows']}")
if isinstance(err['top_messages'], pd.DataFrame) and not err['top_messages'].empty:
    top_msgs = err['top_messages']
    lines.append("  Top messages:")
    for _, m in top_msgs.iterrows():
        lines.append(f"    - {m['message']}: {int(m['count'])}")
else:
    lines.append("  No error_log records available.")

lines.append("")
lines.append("Key takeaways:")
lines.append("- The most prominent log anomalies are in log.row_count for ts-travel-service, ts-route-service, and ts-order-service.")
lines.append("  Each exceeded its global P95 twice within the window and show very large summed row counts (e.g., travel ~39934, route ~23407, order ~9016).")
lines.append("- There are no log.error_count spikes (sums are zero) and no raw error_log records to inspect.")
lines.append("- Interpretation: the logs indicate strong traffic/row_count spikes (likely increased request volume) for travel/route/order during the incident window; investigate upstream traffic patterns, load balancers, or client bursts. Correlate with earlier metric and trace anomalies (latency and inter-service duration spikes).")

summary_text = "\n".join(lines)
summary_text
```

The original code execution output of IPython Kernel is also provided below for reference:

{'group_summary':              cmdb_id       log_name  total_points  p95_value  window_points  count_exceed   earliest_breach_utc  sum_value_in_window  max_value_in_window
0  ts-travel-service  log.row_count            25     1997.6             25             2  2024-01-24T07:36:00Z                39934                 2043
1   ts-route-service  log.row_count            25     1034.8             25             2  2024-01-24T07:46:00Z                23407                 1041
2   ts-order-service  log.row_count            25      460.4             25             2  2024-01-24T07:36:00Z                 9016                  485, 'service_aggregates':              cmdb_id  sum_log_error_count  sum_log_row_count earliest_nonzero_error_count_utc
0    ts-auth-service                  0.0             7556.0                             None
1   ts-order-service                  0.0             9016.0                             None
2   ts-route-service                  0.0            23407.0                             None
3   ts-train-service                  0.0                0.0                             None
4  ts-travel-service                  0.0            39934.0                             None, 'error_logs': {'total_rows': 0, 'top_messages': Empty DataFrame
Columns: [message, count]
Index: []}}```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse existing metric_df and trace_df if available; otherwise load and parse
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    if metric_df['timestamp'].dtype != 'datetime64[ns, UTC]':
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    if trace_df['timestamp'].dtype != 'datetime64[ns, UTC]':
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Parameters
component = 'ts-train-service'
window_start = pd.to_datetime("2024-01-24 07:27:12", utc=True)
window_end   = pd.to_datetime("2024-01-24 07:57:12", utc=True)

# 1) Metrics for ts-train-service: compute global P95 per kpi_name (full series)
metric_comp = metric_df[metric_df['cmdb_id'] == component].copy()
group_keys_m = ['cmdb_id', 'kpi_name']
global_metric_stats = (
    metric_comp
    .groupby(group_keys_m)['value']
    .agg(total_points='count', p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)

# Filter metric rows to incident window (for this component)
metric_window = metric_comp[(metric_comp['timestamp'] >= window_start) & (metric_comp['timestamp'] <= window_end)].copy()

# Merge global p95 into window rows and compute exceedances
metric_window = metric_window.merge(global_metric_stats, on=group_keys_m, how='left')
metric_window['exceed'] = metric_window['value'] > metric_window['p95_value']

# Aggregate window-level metrics
metric_window_agg = (
    metric_window
    .groupby(group_keys_m)
    .agg(
        window_points=('value', 'count'),
        count_exceed=('exceed', 'sum'),
        earliest_breach_ts=('timestamp', lambda s: s[metric_window.loc[s.index, 'exceed']].min() if s.any() else pd.NaT),
        max_value_in_window=('value', 'max')
    )
    .reset_index()
)

# Note: earliest_breach_ts calculation above uses indexing; simpler reliable approach:
if not metric_window[metric_window['exceed']].empty:
    m_breach = metric_window[metric_window['exceed']].groupby(group_keys_m).agg(
        count_exceed=('exceed', 'size'),
        earliest_breach_ts=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        window_points=('value', 'count')
    ).reset_index()
else:
    m_breach = pd.DataFrame(columns=group_keys_m + ['count_exceed','earliest_breach_ts','max_value_in_window','window_points'])

# Merge global stats with breach/window aggregates (prefer using m_breach for correctness)
metric_result = global_metric_stats.merge(m_breach, on=group_keys_m, how='left')

# Fill NaNs and compute percent_over_p95
metric_result['window_points'] = metric_result['window_points'].fillna(0).astype(int)
metric_result['count_exceed'] = metric_result['count_exceed'].fillna(0).astype(int)
metric_result['max_value_in_window'] = metric_result['max_value_in_window']  # keep NaN if none
# earliest_breach_ts stays as Timestamp or NaT

def pct_over_p95(row):
    p95 = row['p95_value']
    mv = row['max_value_in_window']
    if pd.isna(p95) or pd.isna(mv):
        return np.nan
    if p95 == 0.0:
        return np.inf if mv > 0 else 0.0
    return (mv - p95) / p95 * 100.0

metric_result['percent_over_p95'] = metric_result.apply(pct_over_p95, axis=1)

# Filter to count_exceed > 0, format earliest timestamp to ISO, select columns, sort by percent_over_p95 desc
metric_out = metric_result[metric_result['count_exceed'] > 0].copy()
metric_out['earliest_breach_utc'] = metric_out['earliest_breach_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
metric_out = metric_out[['cmdb_id','kpi_name','total_points','p95_value','window_points',
                         'count_exceed','earliest_breach_utc','max_value_in_window','percent_over_p95']]
metric_out = metric_out.sort_values('percent_over_p95', ascending=False).reset_index(drop=True).head(20)

# 2) Traces involving ts-train-service: compute global P95 per (cmdb_id, trace_name) over full trace series
pattern_to = 'to_ts-train-service'
pattern_from = 'from_ts-train-service'
# Identify relevant trace groups in full series (we compute global P95 across full trace_df)
trace_relevant = trace_df[trace_df['trace_name'].str.contains(pattern_to) | trace_df['trace_name'].str.contains(pattern_from)].copy()
group_keys_t = ['cmdb_id','trace_name']
global_trace_stats = (
    trace_relevant
    .groupby(group_keys_t)['value']
    .agg(total_points='count', p95_value=lambda s: s.quantile(0.95))
    .reset_index()
)

# Filter trace rows to the incident window and relevant trace_names
trace_window = trace_relevant[(trace_relevant['timestamp'] >= window_start) & (trace_relevant['timestamp'] <= window_end)].copy()
# Merge p95 into window rows and compute exceedances
trace_window = trace_window.merge(global_trace_stats, on=group_keys_t, how='left')
trace_window['exceed'] = trace_window['value'] > trace_window['p95_value']

# Aggregate breaches
if not trace_window[trace_window['exceed']].empty:
    t_breach = trace_window[trace_window['exceed']].groupby(group_keys_t).agg(
        count_exceed=('exceed', 'size'),
        earliest_breach_ts=('timestamp', 'min'),
        max_value_in_window=('value', 'max'),
        window_points=('value', 'count')
    ).reset_index()
else:
    t_breach = pd.DataFrame(columns=group_keys_t + ['count_exceed','earliest_breach_ts','max_value_in_window','window_points'])

trace_result = global_trace_stats.merge(t_breach, on=group_keys_t, how='left')
trace_result['window_points'] = trace_result['window_points'].fillna(0).astype(int)
trace_result['count_exceed'] = trace_result['count_exceed'].fillna(0).astype(int)

trace_result['percent_over_p95'] = trace_result.apply(pct_over_p95, axis=1)
# Filter to count_exceed > 0
trace_out = trace_result[trace_result['count_exceed'] > 0].copy()
trace_out['earliest_breach_utc'] = trace_out['earliest_breach_ts'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# Select requested columns
trace_out = trace_out[['cmdb_id','trace_name','total_points','p95_value','window_points',
                       'count_exceed','earliest_breach_utc','max_value_in_window','percent_over_p95']]

# Sort by earliest_breach_utc ascending (convert back to datetime for sorting), then percent_over_p95 desc
trace_out['earliest_breach_dt'] = pd.to_datetime(trace_out['earliest_breach_utc'], utc=True)
trace_out = trace_out.sort_values(['earliest_breach_dt','percent_over_p95'], ascending=[True, False]).reset_index(drop=True).drop(columns=['earliest_breach_dt']).head(20)

# 3) Combined earliest breach UTC among all ts-train-service metric and trace groups
# Collect earliest timestamps from metric_out and trace_out (they are ISO strings or NaN)
metric_earlies = pd.to_datetime(metric_out['earliest_breach_utc'], utc=True, errors='coerce')
trace_earlies = pd.to_datetime(trace_out['earliest_breach_utc'], utc=True, errors='coerce')
all_earlies = pd.Series(list(metric_earlies.dropna()) + list(trace_earlies.dropna()))
earliest_overall = all_earlies.min() if not all_earlies.empty else None
earliest_overall_iso = earliest_overall.strftime('%Y-%m-%dT%H:%M:%SZ') if pd.notna(earliest_overall) else None

# Assemble final compact result dict
result = {
    'train_metric_breaches': metric_out,   # DataFrame (compact)
    'train_trace_breaches': trace_out,     # DataFrame (compact)
    'earliest_breach_utc': earliest_overall_iso
}

result

```
Out[6]:
```
ts-train-service is the probable root cause and the anomalous activity began at 2024-01-24T07:30:00Z (UTC).

Key evidence:
- Metrics (ts-train-service):
  - Multiple KPIs breached their global P95 during the incident window. The earliest breaches for metrics are at 2024-01-24T07:30:00Z.
  - Biggest relative metric spikes (percent over global P95):
    - latency-90: ~261% over P95 (max in window ≈ 0.0916, p95 ≈ 0.02535), earliest breach 07:30:00Z.
    - latency-50: ~150% over P95, earliest breach 07:30:00Z.
    - cpu: ~66% over P95, earliest breach 07:30:00Z.
    - diskio: ~41% over P95 (earliest breach 07:32:00Z).
  - Several other KPIs (workload, socket, mem) also had exceedances later in the window.

- Traces (involving ts-train-service):
  - Traces to/from ts-train-service show major duration spikes with earliest breaches at 2024-01-24T07:30:00Z:
    - trace.from_ts-train-service.duration_p95 and trace.to_ts-train-service.duration_p95 (for ts-train-service) both exceeded global P95 (~64x percent over P95 in duration_p95 entries for train).
    - Multiple calling services (ts-travel-service, ts-basic-service, ts-travel2-service, etc.) show increased durations toward ts-train-service at ~07:30–07:34Z.
  - Trace row_count entries for ts-train-service also exceeded P95 (breaches later in window ~07:46Z), indicating elevated request volumes interacting with train.

- Combined earliest breach across metrics and traces: 2024-01-24T07:30:00Z (UTC).

Interpretation and recommendation:
- There is a clear, correlated spike starting at 07:30:00Z impacting ts-train-service: high latency (both p50/p90), increased CPU and disk I/O, and corresponding cross-service trace latency increases.
- This pattern indicates ts-train-service likely experienced a performance/availability degradation around 07:30Z that propagated latency to callers.
- Recommended next steps: investigate ts-train-service around 07:30Z — check service logs, recent deployments, GC/heap/threads, CPU/disk contention, scaling/instance count, and upstream traffic patterns. Correlate with network and infrastructure events.

The original code execution output of IPython Kernel is also provided below for reference:

{'train_metric_breaches':             cmdb_id    kpi_name  total_points     p95_value  window_points  count_exceed   earliest_breach_utc  max_value_in_window  percent_over_p95
0  ts-train-service  latency-90            25  2.535307e-02              2             2  2024-01-24T07:30:00Z         9.161798e-02        261.368366
1  ts-train-service  latency-50            25  8.827814e-03              2             2  2024-01-24T07:30:00Z         2.210917e-02        150.448939
2  ts-train-service         cpu            25  1.159766e+01              2             2  2024-01-24T07:30:00Z         1.928361e+01         66.271586
3  ts-train-service      diskio            25  6.419425e+04              2             2  2024-01-24T07:32:00Z         9.048346e+04         40.952593
4  ts-train-service    workload            25  2.202492e+01              2             2  2024-01-24T07:37:00Z         2.233222e+01          1.395223
5  ts-train-service      socket            25  1.638000e+01              2             2  2024-01-24T07:44:00Z         1.648333e+01          0.630851
6  ts-train-service         mem            25  2.501492e+08              2             2  2024-01-24T07:53:00Z         2.501632e+08          0.005611, 'train_trace_breaches':                cmdb_id                                 trace_name  total_points    p95_value  window_points  count_exceed   earliest_breach_utc  max_value_in_window  percent_over_p95
0     ts-train-service   trace.from_ts-train-service.duration_p95            25     0.002163              2             2  2024-01-24T07:30:00Z             0.003563         64.674423
1     ts-train-service     trace.to_ts-train-service.duration_p95            25     0.002163              2             2  2024-01-24T07:30:00Z             0.003563         64.674423
2    ts-travel-service    trace.to_ts-train-service.duration_mean            25     0.010595              2             2  2024-01-24T07:30:00Z             0.012072         13.936591
3     ts-basic-service    trace.to_ts-train-service.duration_mean            25     0.010408              2             2  2024-01-24T07:30:00Z             0.011750         12.899747
4   ts-travel2-service    trace.to_ts-train-service.duration_mean            25     0.012462              2             2  2024-01-24T07:30:00Z             0.013096          5.088477
5     ts-train-service  trace.from_ts-train-service.duration_mean            25     0.001761              2             2  2024-01-24T07:30:00Z             0.001840          4.445032
6     ts-train-service    trace.to_ts-train-service.duration_mean            25     0.001761              2             2  2024-01-24T07:30:00Z             0.001840          4.445032
7   ts-travel2-service     trace.to_ts-train-service.duration_p95            25     0.066368              2             2  2024-01-24T07:31:00Z             0.073943         11.414051
8     ts-basic-service     trace.to_ts-train-service.duration_p95            25     0.038544              2             2  2024-01-24T07:34:00Z             0.057927         50.286009
9    ts-travel-service     trace.to_ts-train-service.duration_p95            25     0.059435              2             2  2024-01-24T07:34:00Z             0.069384         16.739671
10  ts-travel2-service        trace.to_ts-train-service.row_count            25   300.600000              2             2  2024-01-24T07:34:00Z           305.000000          1.463739
11   ts-travel-service        trace.to_ts-train-service.row_count            25   244.600000              2             2  2024-01-24T07:36:00Z           247.000000          0.981194
12    ts-basic-service        trace.to_ts-train-service.row_count            25   142.400000              2             2  2024-01-24T07:46:00Z           147.000000          3.230337
13    ts-train-service      trace.from_ts-train-service.row_count            25  2591.200000              2             2  2024-01-24T07:46:00Z          2640.000000          1.883297
14    ts-train-service        trace.to_ts-train-service.row_count            25  2591.200000              2             2  2024-01-24T07:46:00Z          2640.000000          1.883297, 'earliest_breach_utc': '2024-01-24T07:30:00Z'}```
```